# Chapter 12. Graph Neural Networks for Chemistry
## Part 2. Message-passing architectures: from equations to tensors

In [Part 1](Chapter12_Part1.ipynb), atoms and bonds became tensors. We now make those tensors communicate: an atom receives a message from each bonded neighbor, updates its hidden vector, and contributes to a molecular prediction. The exact message, aggregation, update, and readout rules define the architecture.

### Learning objectives

- Map a message-passing equation to source/destination indexing and PyTorch operations.
- Make bond type change the transformation of a neighbor's hidden state.
- Distinguish neighborhood aggregation, an atom update, and molecular readout.
- Check shapes, gradients, atom relabeling, edge ordering, and isolation between graphs in a batch.
- Explain what depth, residual connections, and directed-bond states change.
- Compare GCN, GIN, GAT, and D-MPNN without treating their names as performance rankings.

**Scope:** a few embedded molecules, one CPU thread, and small untrained networks. All scores are arbitrary architecture diagnostics, not predicted chemical properties. [Part 3](Chapter12_Part3.ipynb) supplies the measured-data training and evaluation workflow. This notebook runs independently and needs no graph-learning package, downloaded model, or earlier output file.

Use a fresh kernel in the [course environment](Readme.md). The first cell selects the supported sequential MKL runtime before scientific imports and static notebook plots. Run it before the later cells.

### Start here: messages are vectors of numbers

Think of each atom as holding a small notebook of numbers. In one round it receives copies of its neighbors' numbers, transforms them using a shared rule, adds the incoming messages, and updates its own numbers. *Shared* means that the same parameters are used at every atom; it does not mean all atoms produce the same output.

Read $h_i^{(t)}$ as “atom $i$'s vector after round $t$,” $\mathcal N(i)$ as “the neighbors of $i$,” and $\sum$ as “add these vectors.” A weight matrix mixes channels; a nonlinear function lets the update express more than a linear combination. A **forward pass** computes a prediction. **Training** later adjusts the weights using labeled examples.

**First pass:** do the three-atom calculation below, then trace message → aggregation → update → readout. **Deeper pass:** finite-difference gradients, reverse-edge exclusion, and architecture equations. The scores in this part are deliberately untrained; [Part 7](Chapter12_Part7.ipynb) trains an actual PyG model on measurements.

In [ ]:
import os
os.environ["MKL_THREADING_LAYER"] = "SEQUENTIAL"

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
from IPython.display import display, Image
import torch
from torch import nn
from rdkit import Chem, rdBase
from rdkit.Chem.Draw import rdMolDraw2D

OUT = Path("outputs/chapter12_part2")
OUT.mkdir(parents=True, exist_ok=True)
SEED = 1202
torch.manual_seed(SEED)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
DTYPE = torch.float64
DEVICE = torch.device("cpu")
print(f"PyTorch {torch.__version__}; RDKit {rdBase.rdkitVersion}; CPU threads={torch.get_num_threads()}")

### 12.2.1. Keep the graph convention explicit

Let $N$ be the total atoms in a batch, $B$ the total undirected bonds, $E=2B$ the directed edges, and $G$ the number of molecules. Our tensor names follow Part 1:

| Tensor | Shape | Meaning |
|---|---|---|
| `x` | $(N,F)$ | Atom feature rows |
| `edge_index` | $(2,E)$ | Row 0 is source; row 1 is destination |
| `edge_attr` | $(E,4)$ | Single, double, triple, aromatic bond indicators |
| `reverse_edge` | $(E,)$ | Index of the same bond in the opposite direction |
| `batch` | $(N,)$ | Which molecule owns each atom |
| `ptr` | $(G+1,)$ | Start/end offsets for each molecule's atom rows |

To keep the architecture visible, this lesson uses a compact subset of Part 1's encoding: C/N/O/other element indicators, attached-H count divided by 4, formal charge, aromaticity, and ring membership. These are fixed feature definitions, not fitted scaling statistics. The unknown-element category merges distinct elements, so it cannot support claims about chemistry outside the selected examples.

All inputs here are connected molecules with implicit hydrogen nodes omitted. The encoder rejects explicit H atoms, isotopic labels, radicals, and disconnected inputs. It also **omits stereochemistry and 3D geometry**: use richer features when the task requires these distinctions. An architecture cannot recover information erased by its input representation. See [RDKit atom and bond APIs](https://www.rdkit.org/docs/source/rdkit.Chem.rdchem.html).

In [ ]:
ELEMENTS = [6, 7, 8]
NODE_FEATURES = ["C", "N", "O", "other_element", "attached_H/4", "formal_charge", "aromatic", "in_ring"]
BOND_TYPES = [Chem.BondType.SINGLE, Chem.BondType.DOUBLE,
              Chem.BondType.TRIPLE, Chem.BondType.AROMATIC]
BOND_NAMES = ["single", "double", "triple", "aromatic"]

def pack_graphs(molecules):
    if not molecules:
        raise ValueError("Supply at least one molecule.")
    features, edges, edge_features, reverse, graph_ids = [], [], [], [], []
    ptr = [0]
    for graph_id, mol in enumerate(molecules):
        if mol is None or mol.GetNumAtoms() == 0 or len(Chem.GetMolFrags(mol)) != 1:
            raise ValueError("Each input must be a nonempty connected parsed molecule.")
        for atom in mol.GetAtoms():
            if atom.GetAtomicNum() == 1 or atom.GetIsotope() or atom.GetNumRadicalElectrons():
                raise ValueError("This compact lesson excludes explicit H nodes, isotopes, and radicals.")
            number = atom.GetAtomicNum()
            features.append([float(number == z) for z in ELEMENTS] + [float(number not in ELEMENTS),
                             atom.GetTotalNumHs()/4, float(atom.GetFormalCharge()),
                             float(atom.GetIsAromatic()), float(atom.IsInRing())])
            graph_ids.append(graph_id)
        for bond in mol.GetBonds():
            kind = bond.GetBondType()
            if kind not in BOND_TYPES:
                raise ValueError("Only single/double/triple/aromatic bonds are encoded here.")
            a, b = bond.GetBeginAtomIdx()+ptr[-1], bond.GetEndAtomIdx()+ptr[-1]
            edge_id = len(edges)
            edges.extend([(a, b), (b, a)])
            encoded = [float(kind == candidate) for candidate in BOND_TYPES]
            edge_features.extend([encoded, encoded])
            reverse.extend([edge_id+1, edge_id])
        ptr.append(ptr[-1]+mol.GetNumAtoms())
    graph = {"x": torch.tensor(features, dtype=DTYPE, device=DEVICE),
             "edge_index": torch.tensor(edges, dtype=torch.long, device=DEVICE).reshape(-1, 2).T,
             "edge_attr": torch.tensor(edge_features, dtype=DTYPE, device=DEVICE).reshape(-1, 4),
             "reverse_edge": torch.tensor(reverse, dtype=torch.long, device=DEVICE),
             "batch": torch.tensor(graph_ids, dtype=torch.long, device=DEVICE),
             "ptr": torch.tensor(ptr, dtype=torch.long, device=DEVICE)}
    src, dst = graph["edge_index"]
    rev = graph["reverse_edge"]
    assert torch.equal(graph["batch"][src], graph["batch"][dst])
    assert torch.equal(graph["edge_index"][:, rev], graph["edge_index"].flip(0))
    assert torch.equal(rev[rev], torch.arange(len(rev), device=DEVICE))
    return graph

SMILES = ["CC(=O)O", "CCO", "N", "c1ccccc1", "O"]
molecules = [Chem.MolFromSmiles(s) for s in SMILES]
batch_graph = pack_graphs(molecules)
acid = molecules[0]
acid_graph = pack_graphs([acid])
display(pd.DataFrame(acid_graph["x"].numpy(), columns=NODE_FEATURES).rename_axis("atom_index"))
print("Batch shapes:", {key: tuple(value.shape) for key, value in batch_graph.items()})

draw_mol = Chem.Mol(acid)
for atom in draw_mol.GetAtoms():
    atom.SetProp("atomNote", str(atom.GetIdx()))
drawer = rdMolDraw2D.MolDraw2DCairo(450, 230)
drawer.DrawMolecule(draw_mol, legend="Acetic acid: atom indices are tensor addresses")
drawer.FinishDrawing()
image_bytes = drawer.GetDrawingText()
(OUT / "indexed_acetic_acid.png").write_bytes(image_bytes)
display(Image(data=image_bytes))

### A three-atom calculation you can do before coding a network

Use ethanol's C–C–O chain and assign the illustrative numbers $[1,1,2]$. They are one-channel input features, **not atomic numbers or learned properties**. Set the rule to “new value = old value + the sum of neighbors' old values.” All atoms update simultaneously:

$$h_i^{(t+1)}=h_i^{(t)}+\sum_{j\in\mathcal N(i)}h_j^{(t)}.$$

After one round, the middle carbon has $1+1+2=4$, while the left carbon has $1+1=2$. After two rounds, information from oxygen can reach the left carbon. Predict the other numbers before running. This is a useful unit example when a research implementation gives unexpected predictions: first verify where messages go, before blaming a training algorithm.

In [ ]:
demo_adjacency = np.array([[0,1,0], [1,0,1], [0,1,0]], dtype=float)
demo_states = [np.array([1.,1.,2.])]
for _ in range(2):
    demo_states.append(demo_states[-1] + demo_adjacency @ demo_states[-1])
np.testing.assert_array_equal(demo_states[1], [2,4,3])
np.testing.assert_array_equal(demo_states[2], [6,9,7])
fig, axes = plt.subplots(1,3,figsize=(11,2.7),layout='constrained')
for t, (ax, values) in enumerate(zip(axes,demo_states)):
    ax.plot([0,1,2], [0,0,0], color='0.5', lw=3, zorder=1)
    ax.scatter([0,1,2], [0,0,0], s=1000, c=['#dceaf5','#dceaf5','#f6d6d2'], zorder=2)
    for atom,(symbol,value) in enumerate(zip(['C0','C1','O2'], values)):
        ax.text(atom,0,f'{symbol}\n{value:g}',ha='center',va='center',zorder=3)
    ax.set(title=f'Round {t}',xlim=(-.5,2.5),ylim=(-.5,.5))
    ax.axis('off')
fig.savefig(OUT / 'three_atom_message_walkthrough.png', dpi=150)
plt.show()

### 12.2.2. Four operations, with different jobs

A general atom-centered message-passing round is

$$m_{j\to i}^{(t)}=M_t(h_i^{(t)},h_j^{(t)},e_{ji}),\qquad
a_i^{(t)}=\operatorname{AGG}_{j\in\mathcal N(i)}m_{j\to i}^{(t)},$$
$$h_i^{(t+1)}=U_t(h_i^{(t)},a_i^{(t)}),\qquad
z_g=\operatorname{POOL}_{i\in g} h_i^{(T)},\quad \hat y_g=R(z_g).$$

1. **Message:** transform a neighbor's state, possibly using both endpoints and the bond.
2. **Neighborhood aggregation:** reduce incoming edge messages to one vector per destination atom.
3. **Update:** combine that aggregate with the atom's own state.
4. **Readout:** pool atom states within each molecule, then map the pooled vector to the requested target.

The same functions are shared across atoms/edges in a round. Our rounds have distinct parameter sets. The general message function may use the destination state; our implementation below uses the source state and bond, with the destination's own state entering the update. This is one instance of the [MPNN framework](https://proceedings.mlr.press/v70/gilmer17a.html).

### A numerical round that can be checked by hand

For this one calculation only, replace atom features with two indicator entries: carbon is $[1,0]$, oxygen is $[0,1]$. Define artificial message matrices $W_{\rm single}=I$ and $W_{\rm double}=\mathrm{diag}(1,2)$, and use $m_{j\to i}=W_bh_j$ without activation.

Acetic acid's carbonyl carbon (atom 1) receives $[1,0]$ from atom 0, $[0,2]$ from atom 2, and $[0,1]$ from atom 3. Its sum is **$[1,3]$**; its mean is **$[1/3,1]$**. The factor 2 is an invented teaching coefficient, not a bond energy, bond strength, or electronic-structure calculation.

In [ ]:
def aggregate(messages, destination, number_of_rows, reduction="sum"):
    result = messages.new_zeros((number_of_rows, messages.shape[1])).index_add(0, destination, messages)
    if reduction == "mean":
        count = torch.bincount(destination, minlength=number_of_rows).to(messages.dtype).unsqueeze(1)
        result = result/count.clamp_min(1)  # Define an empty-neighborhood mean as zero.
    elif reduction != "sum":
        raise ValueError("Supported reductions are sum and mean.")
    return result

worked_h = torch.tensor([[1., 0.], [1., 0.], [0., 1.], [0., 1.]], dtype=DTYPE)
worked_matrices = torch.stack([torch.eye(2, dtype=DTYPE), torch.diag(torch.tensor([1., 2.], dtype=DTYPE)),
                               torch.eye(2, dtype=DTYPE), torch.eye(2, dtype=DTYPE)])
src, dst = acid_graph["edge_index"]
edge_matrices = torch.einsum("ek,kij->eij", acid_graph["edge_attr"], worked_matrices)
worked_messages = torch.bmm(edge_matrices, worked_h[src].unsqueeze(-1)).squeeze(-1)
worked_sum = aggregate(worked_messages, dst, len(worked_h), "sum")
worked_mean = aggregate(worked_messages, dst, len(worked_h), "mean")
torch.testing.assert_close(worked_sum[1], torch.tensor([1., 3.], dtype=DTYPE))
torch.testing.assert_close(worked_mean[1], torch.tensor([1/3, 1.], dtype=DTYPE))
display(pd.DataFrame({"source": src.numpy(), "destination": dst.numpy(),
                      "bond": [BOND_NAMES[i] for i in acid_graph["edge_attr"].argmax(1).tolist()],
                      "message_C": worked_messages[:, 0].numpy(), "message_O": worked_messages[:, 1].numpy()}))
display(pd.DataFrame({"atom": range(4), "sum": worked_sum.tolist(), "mean": worked_mean.tolist()}))

### 12.2.3. Aggregation controls which distinctions survive

Sum, mean, and componentwise maximum are invariant to the ordering of incoming messages, but retain different information. Repeating the same vector changes its sum and leaves its mean and maximum unchanged. Conversely, sum alone can also collide: scalar multisets $\{1,3\}$ and $\{2,2\}$ both sum to 4. Applying a learned nonlinear transform before summation can preserve distinctions that summing raw values loses.

This does not make every finite neural network an injective encoding of every neighborhood. The [GIN analysis](https://arxiv.org/abs/1810.00826) states expressivity results under assumptions on injective multiset functions and readouts; arbitrary learned weights are not automatically guaranteed to satisfy those assumptions.

The reduction also needs an empty-neighborhood policy. With heavy atoms only, water and ammonia have no edges. We define their incoming aggregate as zero and retain their own atom state through the update.

In [ ]:
one_copy = torch.tensor([[1., 2.]], dtype=DTYPE)
two_copies = one_copy.repeat(2, 1)
display(pd.DataFrame({"neighborhood": ["one copy", "two copies"],
                      "sum": [one_copy.sum(0).tolist(), two_copies.sum(0).tolist()],
                      "mean": [one_copy.mean(0).tolist(), two_copies.mean(0).tolist()],
                      "max": [one_copy.amax(0).tolist(), two_copies.amax(0).tolist()]}))
left, right = torch.tensor([1., 3.]), torch.tensor([2., 2.])
assert left.sum() == right.sum() and left.square().sum() != right.square().sum()
print("Raw sum collision:", left.sum().item(), right.sum().item())
print("After the example transform h -> h^2:", left.square().sum().item(), right.square().sum().item())

### 12.2.4. Implement bond-conditioned messages and residual updates

Let the hidden width be $H$. Initialize $h_i^{(0)}=\tanh(W_{\rm in}x_i+b_{\rm in})$. For one-hot bond attributes, use

$$W(e_{ji})=\sum_{k=1}^{4} e_{ji,k}W_k,\qquad
m_{j\to i}=\tanh\left(W(e_{ji})h_j+b_m\right),$$
$$\Delta h_i=\tanh\left(W_u[h_i\Vert a_i]+b_u\right),\qquad
h_i^{\rm new}=h_i+\Delta h_i.$$

Each bond category selects a learned $H\times H$ matrix. The same atom state can therefore send different transformed messages along single and double bonds. These matrices are learned numerical functions, not force-field or quantum-mechanical operators. With richer edge features, an edge network can instead generate or gate message transformations.

The residual addition preserves a direct path from the previous state; it requires the widths to match. Turning it off below returns $\Delta h_i$. Residual connections can help optimization, but do not guarantee informative deep representations or recover missing chemical information.

| Equation operation | Code | Result shape |
|---|---|---|
| Select $W(e)$ for each edge | `einsum("ek,kij->eij", ...)` | $(E,H,H)$ |
| Transform source states | `bmm(W_edge, h[src, :, None])` | $(E,H,1)$ |
| Aggregate at destination | `aggregate(messages, dst, N)` | $(N,H)$ |
| Update each atom | `update(cat([h, incoming], dim=1))` | $(N,H)$ |

This transparent implementation materializes one small matrix per directed edge. Its memory cost scales as $O(EH^2)$; it is suitable here, while larger graphs may benefit from more economical message parameterizations.

In [ ]:
class BondConditionedLayer(nn.Module):
    def __init__(self, hidden, reduction="sum", residual=True):
        super().__init__()
        self.hidden = hidden
        self.reduction = reduction
        self.residual = residual
        self.bond_weight = nn.Parameter(torch.empty(4, hidden, hidden, dtype=DTYPE))
        for matrix in self.bond_weight:
            nn.init.xavier_uniform_(matrix)
        self.message_bias = nn.Parameter(torch.zeros(hidden, dtype=DTYPE))
        self.update = nn.Linear(2*hidden, hidden, dtype=DTYPE)

    def forward(self, h, edge_index, edge_attr):
        src, dst = edge_index
        edge_weight = torch.einsum("ek,kij->eij", edge_attr, self.bond_weight)
        messages = torch.tanh(torch.bmm(edge_weight, h[src].unsqueeze(-1)).squeeze(-1) + self.message_bias)
        incoming = aggregate(messages, dst, len(h), self.reduction)
        delta = torch.tanh(self.update(torch.cat([h, incoming], dim=1)))
        assert messages.shape == (edge_index.shape[1], self.hidden)
        assert delta.shape == h.shape
        return h+delta if self.residual else delta

class MolecularMPNN(nn.Module):
    def __init__(self, node_dim=8, hidden=12, depth=3, aggregation="sum", pooling="sum", residual=True):
        super().__init__()
        self.pooling = pooling
        self.embedding = nn.Linear(node_dim, hidden, dtype=DTYPE)
        self.layers = nn.ModuleList([BondConditionedLayer(hidden, aggregation, residual) for _ in range(depth)])
        self.readout = nn.Sequential(nn.Linear(hidden, hidden, dtype=DTYPE), nn.Tanh(),
                                     nn.Linear(hidden, 1, dtype=DTYPE))

    def forward(self, graph):
        h = torch.tanh(self.embedding(graph["x"]))
        states = [h]
        for layer in self.layers:
            h = layer(h, graph["edge_index"], graph["edge_attr"])
            states.append(h)
        number_of_graphs = len(graph["ptr"])-1
        pooled = aggregate(h, graph["batch"], number_of_graphs, self.pooling)
        score = self.readout(pooled)
        assert score.shape == (number_of_graphs, 1)
        return score, h, pooled, states

CONFIG = {"node_dim": 8, "hidden": 12, "depth": 3, "aggregation": "sum", "pooling": "sum", "residual": True}
torch.manual_seed(SEED)
model = MolecularMPNN(**CONFIG).to(DEVICE)
model.eval()
with torch.inference_mode():
    scores, hidden, pooled, states = model(batch_graph)
assert all(state.shape == (batch_graph["x"].shape[0], CONFIG["hidden"]) for state in states)
assert torch.isfinite(scores).all()
display(pd.DataFrame({"SMILES": SMILES, "untrained_arbitrary_score": scores[:, 0].numpy()}))
print("Shapes: final atoms", tuple(hidden.shape), "pooled graphs", tuple(pooled.shape), "scores", tuple(scores.shape))
print("Trainable parameters:", sum(parameter.numel() for parameter in model.parameters()))

### Readout is not another neighbor update

`aggregate` serves two different reductions: messages grouped by their **destination atom**, and final atom vectors grouped by their **molecule**. Confusing `dst` with `batch` mixes these operations and often produces the wrong number of rows.

Sum pooling can preserve count information; mean pooling removes the overall multiplicity of identical vectors. Neither choice alone guarantees a physically extensive or intensive prediction. A nonlinear readout of a sum is generally not additive over disconnected components. Predicting an additive quantity can motivate a sum of per-atom contributions, but that physical assumption must fit the task and data. Solubility is not an atom-additive quantity by definition.

An output shape of `(G, 1)` is convenient for regression with targets of exactly the same shape. Binary classification can use the same output shape as raw logits with `BCEWithLogitsLoss`; it requires different labels and loss. The numbers above have no chemical units because this model has not been fitted to a target.

### 12.2.5. Check gradients without pretending to train a predictor

Backpropagation must reach the embedding, bond matrices, update, and readout. We differentiate the sum of the arbitrary scores and compare one double-bond matrix derivative against central finite differences. This is an implementation check, not supervised learning or a performance estimate.

In [ ]:
model.zero_grad(set_to_none=True)
diagnostic_scalar = model(acid_graph)[0].sum()
diagnostic_scalar.backward()
assert all(parameter.grad is not None and torch.isfinite(parameter.grad).all()
           for parameter in model.parameters())
selected_parameter = model.layers[0].bond_weight
selected_entry = (1, 0, 0)  # Double-bond matrix, one output/input component.
autograd_value = selected_parameter.grad[selected_entry].item()
step = 1e-5
with torch.no_grad():
    original_value = selected_parameter[selected_entry].item()
    try:
        selected_parameter[selected_entry] = original_value+step
        plus = model(acid_graph)[0].sum().item()
        selected_parameter[selected_entry] = original_value-step
        minus = model(acid_graph)[0].sum().item()
    finally:
        selected_parameter[selected_entry] = original_value
finite_difference = (plus-minus)/(2*step)
np.testing.assert_allclose(autograd_value, finite_difference, rtol=1e-5, atol=1e-8)
assert abs(autograd_value) > 1e-8
display(pd.DataFrame({"derivative": ["autograd", "central finite difference"],
                      "value": [autograd_value, finite_difference]}))
model.zero_grad(set_to_none=True)

### 12.2.6. Test symmetry and graph isolation

Atom relabeling should reorder atom embeddings (**equivariance**) and preserve a molecular scalar (**invariance**). Edge-list order should also be irrelevant. A disconnected batch must produce the same scores as evaluating its molecules separately; changing one graph must not affect the others.

These properties follow from shared functions and symmetric reductions, up to floating-point summation differences. Numerical tests catch indexing errors in an implementation; they do not show that distinct molecules always receive distinct representations. The tests use double precision and small CPU tensors for tight tolerances.

In [ ]:
permutation = [3, 1, 0, 2]  # New atom i is original atom permutation[i].
renumbered_graph = pack_graphs([Chem.RenumberAtoms(acid, permutation)])
with torch.inference_mode():
    original_score, original_h, _, original_states = model(acid_graph)
    permuted_score, permuted_h, _, permuted_states = model(renumbered_graph)
    separate_scores = torch.cat([model(pack_graphs([mol]))[0] for mol in molecules], dim=0)
    batched_scores = model(batch_graph)[0]
torch.testing.assert_close(original_score, permuted_score, atol=1e-10, rtol=1e-10)
for before, after in zip(original_states, permuted_states):
    torch.testing.assert_close(after, before[permutation], atol=1e-10, rtol=1e-10)
torch.testing.assert_close(separate_scores, batched_scores, atol=1e-10, rtol=1e-10)

# Reorder directed edges and update the reverse-edge map consistently.
order = torch.tensor([4, 1, 5, 2, 0, 3], dtype=torch.long)
old_to_new = torch.empty_like(order)
old_to_new[order] = torch.arange(len(order))
shuffled = dict(acid_graph)
shuffled["edge_index"] = acid_graph["edge_index"][:, order]
shuffled["edge_attr"] = acid_graph["edge_attr"][order]
shuffled["reverse_edge"] = old_to_new[acid_graph["reverse_edge"][order]]
with torch.inference_mode():
    torch.testing.assert_close(model(shuffled)[0], original_score, atol=1e-10, rtol=1e-10)

# Deliberate tensor perturbation for isolation, not a chemically valid new molecule.
changed = dict(batch_graph)
changed["x"] = batch_graph["x"].clone()
changed["x"][batch_graph["batch"] == 2, 0] += 0.2
with torch.inference_mode():
    changed_scores = model(changed)[0]
other_graphs = torch.tensor([0, 1, 3, 4])
torch.testing.assert_close(changed_scores[other_graphs], batched_scores[other_graphs], atol=1e-10, rtol=1e-10)
assert abs((changed_scores[2]-batched_scores[2]).item()) > 1e-8
print("Passed: all-layer atom equivariance; graph invariance; edge-order invariance; separate/batched equality; graph isolation.")
print("Single-atom water and ammonia also produced finite scores with empty edge lists.")

### 12.2.7. Depth increases the receptive field, not guaranteed accuracy

In this local architecture, one round carries information across at most one bond. After $T$ rounds, an atom can depend on inputs within $T$ bonds, plus information already present in its initial features. Graph pooling combines local summaries; it does not make each atom's hidden state globally informed.

We perturb one input component at the oxygen end of 1-butanol and compare each atom's hidden vector at depths 0 through 4. This is a controlled tensor experiment with fixed random weights, not a molecular substitution or a chemical sensitivity explanation. Features such as ring membership can already encode some nonlocal information; the locality assertion here is about propagation of this specific changed tensor entry.

Residual connections preserve previous states but do not create a shortcut across distant atoms. Greater depth can encounter **oversmoothing** (node states losing useful distinctions) or **oversquashing** (too much distant information compressed through limited vectors/bottlenecks). They are different limitations, not inevitable outcomes at a particular universal depth. See [Li et al.](https://arxiv.org/abs/1801.07606) and [Alon & Yahav](https://arxiv.org/abs/2006.05205).

In [ ]:
chain_mol = Chem.MolFromSmiles("CCCCO")
chain_graph = pack_graphs([chain_mol])
perturbed_chain = dict(chain_graph)
perturbed_chain["x"] = chain_graph["x"].clone()
source_atom = 4
perturbed_chain["x"][source_atom, 0] += 0.25
torch.manual_seed(SEED+1)
depth_model = MolecularMPNN(hidden=12, depth=4).to(DEVICE).eval()
with torch.inference_mode():
    base_states = depth_model(chain_graph)[3]
    changed_states = depth_model(perturbed_chain)[3]
    differences = torch.stack([(after-before).norm(dim=1) for before, after in zip(base_states, changed_states)])
bond_distance = Chem.GetDistanceMatrix(chain_mol)[source_atom].astype(int)
for depth, difference in enumerate(differences):
    outside = torch.tensor(bond_distance > depth)
    torch.testing.assert_close(difference[outside], torch.zeros_like(difference[outside]), atol=1e-12, rtol=0)
assert torch.all(differences[-1] > 1e-8)

fig, ax = plt.subplots(figsize=(7, 3.7), layout="constrained")
heatmap = ax.imshow(differences.numpy(), origin="lower", aspect="auto", cmap="Blues")
ax.set_xticks(range(5), [f"{i}: {a.GetSymbol()}\n{bond_distance[i]} bonds away" for i, a in enumerate(chain_mol.GetAtoms())])
ax.set_yticks(range(5))
ax.set(xlabel="Atom index and distance from perturbed atom 4", ylabel="Message-passing rounds",
       title="A changed input travels one bond per round")
for depth in range(5):
    for atom_index in range(5):
        value = differences[depth, atom_index].item()
        ax.text(atom_index, depth, f"{value:.2g}" if value else "0", ha="center", va="center",
                color="white" if value > differences.max().item()*0.6 else "black", fontsize=9)
fig.colorbar(heatmap, ax=ax, label="Norm of hidden-vector difference (arbitrary units)")
fig.savefig(OUT / "receptive_field.png", dpi=140, bbox_inches="tight")
plt.show()

### 12.2.8. A directed-bond state is more than two directed edge entries

So far, the persistent hidden states belong to **atoms**. Storing each bond twice only tells the atom model how to send messages both ways. In a directed-bond MPNN, each orientation $i\to j$ instead has its **own persistent hidden state** $q_{i\to j}$.

A common D-MPNN construction initializes a state from the source atom and bond,

$$q_{i\to j}^{(0)}=\phi(W_{\rm in}[x_i\Vert e_{ij}]),\qquad
s_{i\to j}^{(t)}=\sum_{k\in\mathcal N(i)\setminus\{j\}}q_{k\to i}^{(t)},$$
$$q_{i\to j}^{(t+1)}=\phi\left(q_{i\to j}^{(0)}+W_hs_{i\to j}^{(t)}\right).$$

The sum uses states **arriving at the source $i$**, excluding $j\to i$, the reverse of the outgoing edge. This removes an immediate two-edge return $i\to j\to i$ from that propagation rule; it does not remove graph cycles or make the molecule physically directed. The residual above is to the **initial edge state**, unlike our earlier residual to the previous atom state. After the directed-bond rounds, incoming edge states can be aggregated into atom states with atom features, then pooled into a molecular representation.

The following integer-valued edge states make the exclusion easy to audit. They are arbitrary bookkeeping values. Source: [Yang et al.'s D-MPNN paper](https://pmc.ncbi.nlm.nih.gov/articles/PMC6727618/) and [Chemprop's directed-message implementation](https://chemprop.readthedocs.io/en/latest/autoapi/chemprop/nn/message_passing/base/index.html). We demonstrate the key indexing rule rather than reproduce the full Chemprop model or its benchmarks.

In [ ]:
src, dst = acid_graph["edge_index"]
reverse_edge = acid_graph["reverse_edge"]
edge_state = torch.arange(1, len(src)+1, dtype=DTYPE).reshape(-1, 1)
arriving_at_atom = aggregate(edge_state, dst, acid_graph["x"].shape[0], "sum")
all_arriving_at_source = arriving_at_atom[src]
excluding_reverse = all_arriving_at_source-edge_state[reverse_edge]
torch.testing.assert_close(excluding_reverse[:, 0], torch.tensor([0., 10., 7., 0., 5., 0.], dtype=DTYPE))
assert torch.equal(src[reverse_edge], dst) and torch.equal(dst[reverse_edge], src)
display(pd.DataFrame({"outgoing edge": [f"{i} -> {j}" for i, j in zip(src.tolist(), dst.tolist())],
                      "edge state": edge_state[:, 0].tolist(),
                      "all arriving at source": all_arriving_at_source[:, 0].tolist(),
                      "reverse contribution": edge_state[reverse_edge, 0].tolist(),
                      "after reverse exclusion": excluding_reverse[:, 0].tolist()}))
print("For 1 -> 2, incoming 0 -> 1 and 3 -> 1 contribute 1 + 6 = 7; 2 -> 1 is excluded.")

### 12.2.9. Related architectures: compare their actual operations

Here $H$ denotes the matrix of node states, $\tilde A=A+I$ adds self-loops, and $\tilde D$ contains its row sums. The formulas describe common/original variants; later extensions may add bond features, normalization, residuals, or different readouts.

| Family | Characteristic local operation | Chemical modeling implication |
|---|---|---|
| [GCN — Kipf & Welling](https://arxiv.org/abs/1609.02907) | $H'=\sigma(\tilde D^{-1/2}\tilde A\tilde D^{-1/2}HW)$ | Symmetrically normalized neighbor mixing with self-loops. The original rule does not distinguish chemical bond categories automatically. |
| [GIN — Xu et al.](https://arxiv.org/abs/1810.00826) | $h_i'=\mathrm{MLP}((1+\epsilon)h_i+\sum_j h_j)$ | Sum-based multiset processing; its stated expressivity can match 1-WL under appropriate injectivity assumptions. Edge-aware extensions are needed to use bond attributes explicitly. |
| [GAT — Veličković et al.](https://arxiv.org/abs/1710.10903) | $h_i'=\sigma(\sum_j\alpha_{ij}Wh_j)$, with learned scores normalized over each destination's neighbors | Neighbor contributions depend on node states. The original formulation needs extension for explicit bond attributes; attention weights are not automatically causal chemical explanations. |
| [D-MPNN — Yang et al.](https://pmc.ncbi.nlm.nih.gov/articles/PMC6727618/) | Hidden states on directed bonds; exclude the immediate reverse when collecting incoming states | Maintains an orientation-specific context along each bond before atom/molecule readout. This is a representational choice, not a claim that covalent bonds have one-way physical influence. |
| This notebook's MPNN | Learned matrix selected by bond type, sum/mean aggregate, residual atom update | Makes bond conditioning and every tensor reduction explicit; no benchmark superiority is claimed. |

**1-WL** is the one-dimensional Weisfeiler–Lehman graph-refinement procedure. Matching that test does not solve general graph isomorphism: some different graphs remain indistinguishable. More expressive local aggregation also cannot restore unencoded chirality, isotope, conformer, or experimental context.

Architectures are one part of an experiment. Compare models using the same defined targets, chemical splits, preprocessing rules, tuning budget, and meaningful baselines. Parameter count, depth, or an attention mechanism alone cannot establish better molecular generalization. Part 3 puts a small architecture into a measured-solubility workflow; Part 4 examines failure modes and interpretation.

### 12.2.10. Moving these ideas into a graph library

The same concepts appear in PyTorch Geometric (PyG). Its [introductory documentation](https://pytorch-geometric.readthedocs.io/en/latest/get_started/introduction.html) maps naturally to this chapter:

| Our implementation | PyG concept |
| --- | --- |
| Graph dictionary with `x`, `edge_index`, and `edge_attr` | `Data` object with the same tensor attributes |
| Concatenate graphs and offset their edges | Graph `DataLoader` and `Batch` |
| Node-to-graph membership | `batch` vector |
| Source messages, destination aggregation, then update | `MessagePassing` layer |
| Reduce atom embeddings per molecule | Graph pooling using the membership vector |

Read PyG's [custom message-passing guide](https://pytorch-geometric.readthedocs.io/en/latest/notes/create_gnn.html) when adapting a layer. Check edge direction, automatic self-loops, normalization, and whether the chosen operator actually consumes `edge_attr`. Keep the permutation and batching checks when changing implementations. A library reduces plumbing; it does not choose a chemically appropriate representation or evaluation split for you.

This part implements the operations directly in PyTorch. [Part 6](Chapter12_Part6.ipynb) now implements this bridge with actual `Data`, `Batch`, `DataLoader`, `MessagePassing`, `GCNConv`, and `GINEConv`; Parts 7–8 train and assess PyG models on measured data. Install the complete course requirements before those lessons.

In [ ]:
record = {"scope": "Untrained architecture, gradient, indexing, and locality diagnostics; no property benchmark",
          "seed": SEED, "torch_version": str(torch.__version__), "rdkit_version": rdBase.rdkitVersion,
          "dtype": "float64", "device": "cpu", "node_features": NODE_FEATURES,
          "bond_features": BOND_NAMES, "architecture": CONFIG, "molecules": SMILES,
          "gradient_check": {"parameter": "layer0 double-bond matrix[0,0]",
                             "autograd": autograd_value, "finite_difference": finite_difference},
          "checks": ["worked aggregation", "parameter gradient", "atom equivariance", "edge-order invariance",
                     "batch equivalence", "graph isolation", "empty edge lists", "finite receptive field",
                     "directed reverse-edge exclusion"]}
(OUT / "architecture_checks.json").write_text(json.dumps(record, indent=2)+"\n", encoding="utf-8")
print("Saved architecture diagnostics and figures under", OUT)

### Exercises

1. For 20 atoms, 18 undirected bonds, hidden width 12, and a batch of four molecules, give the shapes of edge indices, messages, aggregated atom messages, pooled graph vectors, and scalar regression outputs.
2. Recalculate the incoming sum and mean at acetic acid atom 1. Which tensor supplies the destination of each message?
3. Give a pair of neighborhoods with identical means but different sums, and a pair with identical raw sums. Why is a nonlinear transform before a sum useful?
4. What would change if the four bond matrices were forced to be equal? Would our feature encoder still distinguish all chemical bonds?
5. Why are a neighbor aggregate and molecular pooling different operations even though both use `index_add`? Is a nonlinear readout of sum-pooled atoms necessarily additive?
6. What do the atom permutation and batch-isolation checks establish? What do they not establish about predictive accuracy or graph distinguishability?
7. In a local three-round model, can the hidden state of an atom depend on a changed input four bonds away? Does adding a residual connection change that distance?
8. For a directed edge $i\to j$, where are its incoming states collected, and which edge is excluded? Contrast the persistent states with an atom-centered MPNN.
9. Does GIN's expressivity result mean every trained GIN distinguishes every pair of molecules? Does a large attention coefficient prove a causal chemical mechanism?

<details><summary>Suggested answers</summary>

1. `(2, 36)`, `(36, 12)`, `(20, 12)`, `(4, 12)`, and `(4, 1)`.
2. Sum $[1,3]$ and mean $[1/3,1]$; `edge_index[1]` is the destination row for each message.
3. One copy of $[1,2]$ versus two copies has the same mean and different sums. $\{1,3\}$ and $\{2,2\}$ share a raw sum. Nonlinear transformed values can separate some such collisions; a particular transform still has limits.
4. Messages would no longer use bond category through the selected transformation. Node features can differ between some bond environments, but they do not generally substitute for explicit bond distinctions.
5. They group by destination atom and by molecule, respectively. A nonlinear graph head generally breaks exact additivity even when its input pooling is additive.
6. They check relabeling symmetry and independence between graphs for these examples. Neither proves that the model is chemically useful, nor that all different graphs have different embeddings.
7. Not through these local rounds; the test is about a changed tensor input, assuming it did not alter other initial features. A local residual does not increase the spatial reach of a round.
8. Collect states arriving at source atom $i$ and exclude $j\to i$. The directed model stores a state for each oriented bond; the atom model stores a state per atom and constructs temporary edge messages.
9. No: the result has assumptions and is bounded by 1-WL; finite learned parameters and input information also matter. Attention weights describe a model's calculation and do not by themselves identify a causal mechanism.

</details>

### Primary references and next steps

- [Gilmer et al., Neural Message Passing for Quantum Chemistry](https://proceedings.mlr.press/v70/gilmer17a.html).
- [Kipf & Welling, GCN](https://arxiv.org/abs/1609.02907); [Xu et al., GIN](https://arxiv.org/abs/1810.00826); [Veličković et al., GAT](https://arxiv.org/abs/1710.10903).
- [Yang et al., Analyzing Learned Molecular Representations for Property Prediction](https://pmc.ncbi.nlm.nih.gov/articles/PMC6727618/).
- [PyTorch indexed addition](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.index_add_.html), [batched matrix multiplication](https://docs.pytorch.org/docs/stable/generated/torch.bmm.html), and [autograd](https://docs.pytorch.org/docs/stable/notes/autograd.html).

[Previous: molecular graphs and batches](Chapter12_Part1.ipynb) · [Next: learning measured solubility](Chapter12_Part3.ipynb) · [Course contents](Readme.md)